# 512890.SH.parquet 前复权流程

在本 Notebook 中完成：读取数据、字段检查、预处理、前复权计算、结果校验、绘图与导出。

## 1) 安装并导入依赖（uv + Python库）

如缺包可先在终端执行：

```bash
uv add pandas pyarrow matplotlib numpy
```

然后导入依赖并设置绘图风格与中文显示。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

plt.style.use("seaborn-v0_8")
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("依赖导入完成")

## 2)~7) 读取、预处理、前复权、校验、绘图与导出

下面一个代码单元依次完成：
- 读取 `512890.SH.parquet` 并检查字段
- 数据预处理（日期索引、排序、去重、缺失值）
- 计算前复权价格（优先 `adj_factor`，否则链式近似）
- 校验结果
- 绘图（原始收盘 vs 前复权收盘）
- 导出结果与图像

In [ ]:
# 2) 读取 parquet 并检查字段
file_path = Path("512890.SH.parquet")
if not file_path.exists():
    raise FileNotFoundError(f"未找到文件: {file_path.resolve()}")

df_raw = pd.read_parquet(file_path)
print("原始数据 shape:", df_raw.shape)
display(df_raw.head())
print("字段列表:", list(df_raw.columns))
print("info:")
df_raw.info()

# 字段标准化映射
column_alias = {
    "trade_date": ["trade_date", "date", "datetime", "dt"],
    "close": ["close", "Close", "收盘价", "收盘"],
    "open": ["open", "Open", "开盘价", "开盘"],
    "high": ["high", "High", "最高价", "最高"],
    "low": ["low", "Low", "最低价", "最低"],
    "adj_factor": ["adj_factor", "adjfactor", "factor", "qfq_factor", "复权因子", "前复权因子"],
}

rename_map = {}
existing_lower_map = {c.lower(): c for c in df_raw.columns}
for target, aliases in column_alias.items():
    for alias in aliases:
        if alias.lower() in existing_lower_map:
            rename_map[existing_lower_map[alias.lower()]] = target
            break

df = df_raw.rename(columns=rename_map).copy()
required = ["trade_date", "close"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"缺少核心字段: {missing_required}")

print("标准化后字段:", list(df.columns))

# 3) 预处理：日期索引、排序、去重、缺失值
if np.issubdtype(df["trade_date"].dtype, np.number):
    df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str), format="%Y%m%d", errors="coerce")
else:
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")

df = df.dropna(subset=["trade_date", "close"]).copy()
for c in ["open", "high", "low", "close", "adj_factor"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 缺失值简单处理：close 用前向填充；OHLC 尽量由 close 补齐
df = df.sort_values("trade_date")
df = df.drop_duplicates(subset=["trade_date"], keep="last")
df["close"] = df["close"].ffill()
for c in ["open", "high", "low"]:
    if c in df.columns:
        df[c] = df[c].fillna(df["close"])

df = df.set_index("trade_date")

# 4) 计算前复权因子与前复权价格
used_factor_col = None
if "adj_factor" in df.columns and df["adj_factor"].notna().sum() > 5:
    used_factor_col = "adj_factor"
    factor = df[used_factor_col].ffill()
    latest_factor = factor.iloc[-1]
    if latest_factor == 0 or pd.isna(latest_factor):
        raise ValueError("最新交易日复权因子无效，无法按 adj_factor 计算前复权")
    scale = factor / latest_factor
    df["close_fq"] = df["close"] * scale
else:
    # 备选：基于 close 收益率链式累乘，令最后一日与原 close 对齐
    ret = df["close"].pct_change().fillna(0.0)
    chain = (1 + ret).cumprod()
    chain = chain / chain.iloc[-1]
    df["close_fq"] = df["close"].iloc[-1] * chain

print("前复权计算完成。使用因子列:", used_factor_col or "链式备选实现")

# 5) 校验前复权结果
print("\n随机抽样对比(最多5行):")
sample_n = min(5, len(df))
display(df[["close", "close_fq"]].dropna().sample(sample_n, random_state=42).sort_index())

last_row = df[["close", "close_fq"]].iloc[-1]
print("最新交易日 close / close_fq:")
print(last_row)
print("差值:", float(last_row["close_fq"] - last_row["close"]))

print("\n描述统计:")
display(df[["close", "close_fq"]].describe())

# 6) 绘图：原始收盘 vs 前复权收盘
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["close"], label="Raw Close", linewidth=1.2, alpha=0.9)
ax.plot(df.index, df["close_fq"], label="QFQ Close", linewidth=1.5)
ax.set_title("512890 原始收盘价 vs 前复权收盘价")
ax.set_xlabel("日期")
ax.set_ylabel("价格")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 局部区间放大（最后120个交易日）
zoom_n = min(120, len(df))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index[-zoom_n:], df["close"].iloc[-zoom_n:], label="Raw Close", linewidth=1.2)
ax.plot(df.index[-zoom_n:], df["close_fq"].iloc[-zoom_n:], label="QFQ Close", linewidth=1.5)
ax.set_title(f"最近{zoom_n}个交易日局部放大")
ax.set_xlabel("日期")
ax.set_ylabel("价格")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 7) 导出结果与图像
output_parquet = Path("512890.SH.qfq.parquet")
output_csv = Path("512890.SH.qfq.csv")
output_png = Path("512890.SH.qfq_vs_raw.png")

df_out = df.reset_index()
df_out.to_parquet(output_parquet, index=False)
df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["close"], label="Raw Close", linewidth=1.2)
ax.plot(df.index, df["close_fq"], label="QFQ Close", linewidth=1.5)
ax.set_title("512890 原始收盘价 vs 前复权收盘价")
ax.set_xlabel("日期")
ax.set_ylabel("价格")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(output_png, dpi=150)
plt.close(fig)

print("\n导出完成:")
print("-", output_parquet.resolve())
print("-", output_csv.resolve())
print("-", output_png.resolve())

print("\n可复现顺序: 依次执行 Notebook 的 Cell 1 -> Cell 4")